In [1]:
# ==========================================
# 🔑 FRED API 키 (입력 필수!)
# ==========================================
FRED_API_KEY = "9c3f2227440e6d8c815f7996a4d253b5"

In [4]:
# ==========================================
# 🔑 FRED API 키 (입력 필수!)
# ==========================================
FRED_API_KEY = "9c3f2227440e6d8c815f7996a4d253b5"

import yfinance as yf
import pandas as pd
import numpy as np
import requests
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("🚀 [Data Factory V3] 고성능 퀀트 센서 26종 통합 수집 및 엔지니어링 시작...")

# 1. 수집 기간 설정
download_start    = '2009-01-01'
actual_start_date = '2010-01-01'
end_date          = '2026-04-02'

# ==========================================
# 1. 자산 가격 및 시장 폭(Breadth) 데이터 수집 (Yahoo)
# ==========================================
assets = ['SPY', 'QQQ', 'EEM', 'TLT', 'IEF', 'LQD', 'SHV', 'GLD', 'DBC', 'VNQ', 'RSP']
df_raw = yf.download(assets, start=download_start, end=end_date)
df_assets = df_raw['Adj Close'] if 'Adj Close' in df_raw.columns else df_raw['Close']
df_assets.dropna(inplace=True)
df_assets.index = pd.to_datetime(df_assets.index).tz_localize(None).normalize()

# ==========================================
# 2. HY Spread 대체 지표 (HYG/LQD 비율)
# FRED BAMLH0A0HYM2 시리즈가 2023년부터만 제공되어
# yfinance HYG/LQD 비율로 대체 (2009년부터 커버)
# ==========================================
hyg_raw   = yf.download('HYG', start=download_start, end=end_date)
lqd_raw   = yf.download('LQD', start=download_start, end=end_date)
hyg_close = hyg_raw['Close'].squeeze()
lqd_close = lqd_raw['Close'].squeeze()
hy_proxy  = (hyg_close / lqd_close)
hy_proxy.name  = 'HY_Spread'
hy_proxy.index = pd.to_datetime(hy_proxy.index).tz_localize(None).normalize()

# df_assets에 HY_Spread 병합
df_assets = pd.concat([df_assets, hy_proxy], axis=1)
print(f"   ✅ HY_Spread 대체 지표 생성 완료 (시작일: {hy_proxy.first_valid_index().date()})")

# ==========================================
# 3. 거시 경제 및 원자재 수집 (Yahoo)
# ==========================================
macro_tickers = {
    '^VIX'    : 'VIX',
    'DX-Y.NYB': 'DXY',
    'KRW=X'   : 'USDKRW',
    'CL=F'    : 'WTI_Oil',
    'HG=F'    : 'Copper',
    'GC=F'    : 'Gold'
}
df_macro_yf_raw = yf.download(list(macro_tickers.keys()), start=download_start, end=end_date)
df_macro_yf     = df_macro_yf_raw['Adj Close'] if 'Adj Close' in df_macro_yf_raw.columns else df_macro_yf_raw['Close']
df_macro_yf.rename(columns=macro_tickers, inplace=True)
df_macro_yf.index = pd.to_datetime(df_macro_yf.index).tz_localize(None).normalize()

# ==========================================
# 4. FRED API 데이터 수집 (HY_Spread 제외)
# ==========================================
def get_fred_api_data(series_id, api_key):
    url = (f"https://api.stlouisfed.org/fred/series/observations"
           f"?series_id={series_id}&api_key={api_key}&file_type=json")
    response = requests.get(url)
    if response.status_code != 200:
        print(f"⚠️ FRED API Error: {series_id}")
        return pd.Series()
    data = response.json()['observations']
    df   = pd.DataFrame(data)
    df['date']  = pd.to_datetime(df['date'])
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    df.set_index('date', inplace=True)
    return df['value']

# HY_Spread는 yfinance로 대체했으므로 제외
fred_series = {
    'DGS10'  : 'US10Y',
    'DGS2'   : 'US2Y',
    'DGS3MO' : 'US3M',
    'WM2NS'  : 'M2_Supply',
    'WALCL'  : 'Fed_Balance',
    'ICSA'   : 'Jobless_Claims'
}
fred_list = []
for series_id, col_name in fred_series.items():
    s      = get_fred_api_data(series_id, FRED_API_KEY)
    s.name = col_name
    fred_list.append(s)

df_macro_fred = pd.concat(fred_list, axis=1).loc[download_start:end_date].ffill()
df_macro_fred.index = pd.to_datetime(df_macro_fred.index).tz_localize(None).normalize()

# ==========================================
# 5. 병합 (outer join + SPY 거래일 기준 필터링)
# ==========================================
df_raw_combined = pd.concat(
    [df_assets, df_macro_yf, df_macro_fred], axis=1, join='outer'
).ffill().bfill()
df_raw_combined.dropna(subset=['SPY'], inplace=True)

# ==========================================
# 6. 26개 퀀트 파생 지표 계산
# ==========================================
df = df_raw_combined.copy()

df['VIX_ret']        = df['VIX'].pct_change()
df['VIX_MA5_Diff']   = df['VIX'] - df['VIX'].rolling(window=5).mean()
df['Volatility_20d'] = df['SPY'].pct_change().rolling(window=20).std() * np.sqrt(252)
df['Vol_Ratio']      = df['Volatility_20d'] / (df['SPY'].pct_change().rolling(window=60).std() * np.sqrt(252))
df['DXY_ret']        = df['DXY'].pct_change()
df['USDKRW_ret']     = df['USDKRW'].pct_change()
df['US10Y_diff']     = df['US10Y'].diff()
df['spread_10y2y']   = df['US10Y'] - df['US2Y']
df['spread_10y3m']   = df['US10Y'] - df['US3M']
df['HY_spread_ret']  = df['HY_Spread'].pct_change()   # 비율 변화로 계산
df['GOLD_ret']       = df['Gold'].pct_change()
df['OIL_ret']        = df['WTI_Oil'].pct_change()
df['Copper_Gold_Ratio'] = df['Copper'] / df['Gold']
df['Market_Breadth'] = df['RSP'] / df['SPY']
df['Equity_vs_Bond'] = df['SPY'] / df['TLT']
df['SPY_ret']        = df['SPY'].pct_change()
df['SPY_Log_Ret']    = np.log(df['SPY'] / df['SPY'].shift(1))
df['SPY_MA20_Diff']  = (df['SPY'] / df['SPY'].rolling(window=20).mean()) - 1
df['MA200_Dist']     = df['SPY'] / df['SPY'].rolling(window=200).mean()

delta = df['SPY'].diff()
gain  = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss  = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs    = gain / loss
df['RSI']  = 100 - (100 / (1 + rs))

exp1       = df['SPY'].ewm(span=12, adjust=False).mean()
exp2       = df['SPY'].ewm(span=26, adjust=False).mean()
df['MACD'] = exp1 - exp2
df['Month'] = df.index.month

df['M2_Growth']          = df['M2_Supply'].pct_change(60)
df['Fed_BS_Growth']      = df['Fed_Balance'].pct_change(60)
df['Jobless_Claims_MA']  = df['Jobless_Claims'].rolling(window=20).mean()

df.dropna(inplace=True)
df = df.loc[actual_start_date:]

# ==========================================
# 7. 피처 스케일링 (26개)
# ==========================================
features_all = [
    "VIX_ret", "VIX", "VIX_MA5_Diff", "Vol_Ratio", "Volatility_20d",
    "DXY_ret", "USDKRW_ret", "DXY",
    "US10Y_diff", "US10Y", "spread_10y2y", "spread_10y3m",
    "GOLD_ret", "OIL_ret",
    "HY_spread_ret", "SPY_ret", "SPY_Log_Ret", "SPY_MA20_Diff", "Equity_vs_Bond",
    "RSI", "MACD", "Month",
    "Copper_Gold_Ratio", "Market_Breadth", "MA200_Dist",
    "M2_Growth", "Fed_BS_Growth", "Jobless_Claims_MA"
]

scaler = StandardScaler()
df_final_scaled = df.copy()
df_final_scaled[features_all] = scaler.fit_transform(df[features_all])

print(f"\n📊 26개 피처 정제 및 스케일링 완료! 최종 데이터 형태: {df_final_scaled.shape}")
print(f"   데이터 기간: {df_final_scaled.index[0].date()} ~ {df_final_scaled.index[-1].date()}")


🚀 [Data Factory V3] 고성능 퀀트 센서 26종 통합 수집 및 엔지니어링 시작...


[*********************100%***********************]  11 of 11 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  6 of 6 completed


   ✅ HY_Spread 대체 지표 생성 완료 (시작일: 2009-01-02)

📊 26개 피처 정제 및 스케일링 완료! 최종 데이터 형태: (5088, 49)
   데이터 기간: 2010-01-01 ~ 2026-04-02


In [3]:
df_final_scaled

,DBC,EEM,GLD,IEF,LQD,QQQ,RSP,SHV,SPY,TLT,...,SPY_ret,SPY_Log_Ret,SPY_MA20_Diff,MA200_Dist,RSI,MACD,Month,M2_Growth,Fed_BS_Growth,Jobless_Claims_MA
2010-01-01,20.814674,29.491636,107.309998,61.396442,57.346958,39.709255,30.289602,88.487404,83.382271,55.978474,...,-0.046661,-0.041711,-0.149054,1.109376,-0.160750,-0.110654,-1.572829,-0.149152,0.002766,0.294541
2010-01-02,20.814674,29.491636,107.309998,61.396442,57.346958,39.709255,30.289602,88.487404,83.382271,55.978474,...,-0.046661,-0.041711,-0.178173,1.085863,0.746969,-0.129668,-1.572829,-0.149152,0.002766,0.289564
2010-01-04,21.338844,30.351509,109.800003,61.548862,57.649811,40.290779,30.787655,88.495468,84.796379,55.928650,...,1.702615,1.689513,0.551916,1.403590,1.104557,-0.107615,-1.572829,0.308399,0.002766,0.284587
2010-01-05,21.364208,30.571804,109.699997,61.819103,57.925125,40.290779,30.963884,88.463310,85.020851,56.289856,...,0.226384,0.230441,0.611018,1.431344,1.166853,-0.086100,-1.572829,0.308399,0.002766,0.279489
2010-01-06,21.744656,30.635763,111.510002,61.569710,57.754471,40.047760,31.078825,88.471329,85.080688,55.536285,...,0.025932,0.030715,0.592674,1.421544,0.918526,-0.069744,-1.572829,0.308399,0.296210,0.274391
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-28,29.100000,55.200001,414.700012,93.976067,106.761131,562.580017,188.460007,110.036087,634.090027,84.985359,...,-0.046661,-0.041711,-1.950816,-1.748071,-1.630562,-3.755251,-0.996089,-0.025481,-0.050895,-0.369778
2026-03-30,29.260000,54.750000,414.579987,94.641647,107.445618,558.280029,188.070007,110.056023,631.969971,86.116646,...,-0.391523,-0.386477,-1.969547,-1.806480,-1.592677,-4.004431,-0.996089,0.453203,-0.050895,-0.371113
2026-03-31,28.950001,56.790001,430.290009,94.810532,108.120193,577.179993,191.919998,110.066002,650.340027,86.027336,...,2.951558,2.907992,-0.596889,-1.303018,-0.738783,-3.655875,-0.996089,0.453203,-0.050895,-0.372327
2026-04-01,28.680000,57.230000,437.820007,94.727798,108.235748,584.309998,192.539993,110.080002,655.239990,85.942650,...,0.730483,0.731010,-0.173964,-1.171410,-0.806562,-3.210537,-0.707719,0.453203,-0.015375,-0.373540


In [5]:
# 데이터 저장
import os
os.makedirs('./데이터 파일', exist_ok=True)

# data_processor.py가 읽는 파일명 그대로 저장
df.to_csv('./데이터 파일/all_etf_price_basic.csv')      # 자산 가격
df_macro_yf.to_csv('./데이터 파일/macro_yfinance_data.csv')  # 거시경제
df_macro_fred.to_csv('./데이터 파일/fred_macro_data.csv')    # FRED 금리

print("✅ CSV 저장 완료!")
print(f"   ./데이터 파일/all_etf_price_basic.csv")
print(f"   ./데이터 파일/macro_yfinance_data.csv")
print(f"   ./데이터 파일/fred_macro_data.csv")

✅ CSV 저장 완료!
   ./데이터 파일/all_etf_price_basic.csv
   ./데이터 파일/macro_yfinance_data.csv
   ./데이터 파일/fred_macro_data.csv
